## Check Environment

In [1]:

import sys
import tensorflow as tf

print(f"Python version: {sys.version}")
print(f"TensorFlow version: {tf.__version__}")


Python version: 3.9.6 (default, Apr 30 2025, 02:07:17) 
[Clang 17.0.0 (clang-1700.0.13.5)]
TensorFlow version: 2.13.0


# Incident Classification with Deep Learning

## 1. Imports and Setup

In [2]:

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords
import re
import nltk
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
import numpy as np # Import numpy if not already imported
from scipy.sparse import hstack, csr_matrix
from sklearn.base import BaseEstimator, TransformerMixin
import sys
import joblib # Import joblib
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
import os


## 2. NLTK Stopwords Download

In [3]:

try:
    stopwords.words('english')
except LookupError:
    nltk.download('stopwords')


## 3. Load Mappings and Data

In [4]:

excel_path = 'data/oiics_201_code_list.xlsx'
sheet_mappings = pd.read_excel(excel_path, sheet_name=['Nature', 'Part', 'Event', 'Source'])

nature_code_mapping = sheet_mappings['Nature'].set_index('CASE_CODE')['CASE_CODE_TITLE'].to_dict()
part_of_body_mapping = sheet_mappings['Part'].set_index('CASE_CODE')['CASE_CODE_TITLE'].to_dict()
event_code_mapping = sheet_mappings['Event'].set_index('CASE_CODE')['CASE_CODE_TITLE'].to_dict()
source_code_mapping = sheet_mappings['Source'].set_index('CASE_CODE')['CASE_CODE_TITLE'].to_dict()

try:
    df = pd.read_excel('data/January2015toApril2024.xlsx')
    print("Data loaded successfully.")
except FileNotFoundError:
    print("Error: The file 'data/January2015toApril2024.xlsx' was not found.")
    print("Please make sure the file path is correct.")


Data loaded successfully.


## 4. Data Cleaning and Preprocessing

In [5]:

df.dropna(subset=['Final Narrative', 'NatureTitle', 'Part of Body Title', 'EventTitle', 'SourceTitle', 'Secondary Source Title', 'Amputation', 'Hospitalized'], inplace=True)
print("Dropped rows with missing values.")

stop_words = set(stopwords.words('english'))
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]
    return ' '.join(tokens)

df['Narrative_Cleaned'] = df['Final Narrative'].apply(clean_text)
print("Text cleaned successfully.")


Dropped rows with missing values.
Text cleaned successfully.


## 5. Feature Engineering

In [6]:

def map_nature_code_refined(code):
    code_str = str(int(code))
    
    while code_str.endswith('0') and len(code_str) > 1:
        code_str = code_str[:-1]

    if len(code_str) >= 4:
        general_code = code_str[:3]
    else:
        general_code = code_str
    
    return nature_code_mapping.get(int(general_code), 'Other')

df['NatureTitle_Generalized'] = df['Nature'].apply(map_nature_code_refined)

def map_part_of_body_code_refined(code):
    code_str = str(int(code))
    
    while code_str.endswith('0') and len(code_str) > 1:
        code_str = code_str[:-1]

    if len(code_str) >= 4:
        general_code = code_str[:3]
    else:
        general_code = code_str
    
    return part_of_body_mapping.get(int(general_code), 'Other')

df['PartofBody_Generalized'] = df['Part of Body'].apply(map_part_of_body_code_refined)

def map_event_code_refined(code):
    code_str = str(int(code))
    
    while code_str.endswith('0') and len(code_str) > 1:
        code_str = code_str[:-1]

    if len(code_str) >= 4:
        general_code = code_str[:3]
    else:
        general_code = code_str
    
    return event_code_mapping.get(int(general_code), 'Other')

df['EventTitle_Generalized'] = df['Event'].apply(map_event_code_refined)

def map_source_code_refined(code):
    code_str = str(int(code))
    
    while code_str.endswith('0') and len(code_str) > 1:
        code_str = code_str[:-1]

    if len(code_str) >= 4:
        general_code = code_str[:3]
    else:
        general_code = code_str
    
    return source_code_mapping.get(int(general_code), 'Other')

df['SourceTitle_Generalized'] = df['Source'].apply(map_source_code_refined)


## 6. Feature and Target Selection

In [7]:

features = ['Narrative_Cleaned', 'NatureTitle_Generalized', 'PartofBody_Generalized', 'EventTitle_Generalized', 'SourceTitle_Generalized']
targets = ['NatureTitle_Generalized', 'PartofBody_Generalized', 'EventTitle_Generalized', 'SourceTitle_Generalized']

y_str = df[targets].copy()

mask = pd.Series(True, index=y_str.index)
for col in targets:
    value_counts = y_str[col].value_counts()
    rare_classes = value_counts[value_counts < 2].index
    if len(rare_classes) > 0:
        mask &= ~y_str[col].isin(rare_classes)

X_filtered = df[features][mask]
y_filtered_str = y_str[mask]
print("Rare classes filtered out.")

y = pd.DataFrame(index=y_filtered_str.index)
label_encoders = {}
for col in targets:
    le = LabelEncoder()
    y[col] = le.fit_transform(y_filtered_str[col])
    label_encoders[col] = le


Rare classes filtered out.


## 7. Train-Test Split

In [8]:

X_train, X_test, y_train, y_test = train_test_split(X_filtered, y, test_size=0.2, random_state=42)
print("Data split into training and testing sets.")


Data split into training and testing sets.


## 8. Custom Transformer and Feature Transformation

In [9]:

class CombinedAttributesAdder(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.vectorizer = TfidfVectorizer(max_features=5000)

    def fit(self, X, y=None):
        self.vectorizer.fit(X['Narrative_Cleaned'])
        return self

    def transform(self, X):
        text_features = self.vectorizer.transform(X['Narrative_Cleaned'])
        return hstack([text_features])

feature_transformer = CombinedAttributesAdder()
X_train_transformed = feature_transformer.fit_transform(X_train)
X_test_transformed = feature_transformer.transform(X_test)
print("Feature transformation complete.")


Feature transformation complete.


## 9. Model Definition

In [10]:

n_classes_nature = len(label_encoders['NatureTitle_Generalized'].classes_)
n_classes_part_of_body = len(label_encoders['PartofBody_Generalized'].classes_)
n_classes_event = len(label_encoders['EventTitle_Generalized'].classes_)
n_classes_source = len(label_encoders['SourceTitle_Generalized'].classes_)

tf.keras.backend.clear_session()

input_shape = X_train_transformed.shape[1]
input_layer = Input(shape=(input_shape,), sparse=True, name='input')
shared_layer1 = Dense(128, activation='relu')(input_layer)
shared_layer2 = Dense(64, activation='relu')(shared_layer1)

output_nature = Dense(n_classes_nature, activation='softmax', name='nature')(shared_layer2)
output_part_of_body = Dense(n_classes_part_of_body, activation='softmax', name='part_of_body')(shared_layer2)
output_event = Dense(n_classes_event, activation='softmax', name='event')(shared_layer2)
output_source = Dense(n_classes_source, activation='softmax', name='source')(shared_layer2)

model = Model(inputs=input_layer, outputs=[output_nature, output_part_of_body, output_event, output_source])


## 10. Model Compilation

In [11]:
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics={'nature': 'accuracy', 'part_of_body': 'accuracy', 'event': 'accuracy', 'source': 'accuracy'})
print("Model compilation complete.")

Model compilation complete.


## 11. Model Training

In [12]:

y_train_list = [y_train[col].values for col in targets]

X_train_transformed_dense = X_train_transformed.toarray()
model.fit(X_train_transformed_dense, y_train_list, epochs=10, batch_size=32, validation_split=0.1)
print("Model training complete.")


Epoch 1/10
803/803 [==============================] - 2s 2ms/step - loss: 10.4166 - nature_loss: 1.6739 - part_of_body_loss: 2.7788 - event_loss: 2.3388 - source_loss: 3.6251 - nature_accuracy: 0.5851 - part_of_body_accuracy: 0.3309 - event_accuracy: 0.4660 - source_accuracy: 0.2395 - val_loss: 7.9346 - val_nature_loss: 1.1967 - val_part_of_body_loss: 2.1879 - val_event_loss: 1.6011 - val_source_loss: 2.9489 - val_nature_accuracy: 0.6778 - val_part_of_body_accuracy: 0.4730 - val_event_accuracy: 0.6311 - val_source_accuracy: 0.3587
Epoch 2/10
803/803 [==============================] - 1s 2ms/step - loss: 6.8896 - nature_loss: 0.9740 - part_of_body_loss: 1.8102 - event_loss: 1.4880 - source_loss: 2.6174 - nature_accuracy: 0.7363 - part_of_body_accuracy: 0.5644 - event_accuracy: 0.6424 - source_accuracy: 0.4308 - val_loss: 6.4347 - val_nature_loss: 0.9405 - val_part_of_body_loss: 1.6496 - val_event_loss: 1.3705 - val_source_loss: 2.4741 - val_nature_accuracy: 0.7546 - val_part_of_body_acc

## 12. Save Model and Artifacts

In [13]:

model.save('deeplearning_model')
print("Deep learning model saved to deeplearning_model.h5")

joblib.dump(feature_transformer, 'deeplearning_feature_transformer.pkl')
joblib.dump(label_encoders, 'deeplearning_label_encoders.pkl')
print("Feature transformer and label encoders saved.")


INFO:tensorflow:Assets written to: deeplearning_model/assets


INFO:tensorflow:Assets written to: deeplearning_model/assets


Deep learning model saved to deeplearning_model.h5
Feature transformer and label encoders saved.


## 13. Make Predictions

In [14]:

X_test_transformed_dense = X_test_transformed.toarray()
y_pred_proba = model.predict(X_test_transformed_dense)
print("Predictions complete.")


223/223 [==============================] - 0s 487us/step
Predictions complete.


## 14. Evaluate Model

In [17]:

y_pred = [np.argmax(proba, axis=1) for proba in y_pred_proba]

with open("classification_reports_deeplearning.txt", "w") as f:
    for i, target_col in enumerate(targets):
        f.write(f"--- Classification Report for {target_col} ---")
        
        y_test_labels = label_encoders[target_col].inverse_transform(y_test[target_col].values)
        y_pred_labels = label_encoders[target_col].inverse_transform(y_pred[i])

        report = classification_report(y_test_labels, y_pred_labels, zero_division=0)
        f.write(report + "")
print("Classification reports have been written to classification_reports_deeplearning.txt")


Classification reports have been written to classification_reports_deeplearning.txt
